In [1]:
!pip install numpyro

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.8/394.8 kB 3.0 MB/s eta 0:00:00


In [2]:
import pymc as pm
import pytensor
import pytensor.tensor as pt
import jax
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from numpy.lib.stride_tricks import sliding_window_view
import matplotlib.pyplot as plt
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import pdist

In [3]:
def extract_hrv_features(serie, window_size=20, window_size_long=40):
    """
    Toma una serie de intervalos RR y los tamaños de ventana para construir
    un DataFrame de características (features) y la variable objetivo (target).
    """
    if window_size_long < window_size:
        raise ValueError("window_size_long debe ser mayor o igual a window_size")

    serie = np.asarray(serie, dtype=float)

    # 1. Generar las ventanas sobre la serie COMPLETA usando la ventana LARGA
    ventanas_long = sliding_window_view(serie, window_size_long)

    # 2. Separar las variables predictoras del target
    X_ventanas_long = ventanas_long[:-1]
    y_target = serie[window_size_long:]

    # 3. Extraer la ventana CORTA (últimos 'window_size' valores)
    X_ventanas_short = X_ventanas_long[:, -window_size:]

    # 4. Calcular las diferencias dentro de cada ventana CORTA
    diffs = np.diff(X_ventanas_short, axis=1)

    # ---- Porta Index ----
    n_above = np.sum(diffs > 0, axis=1)
    n_below = np.sum(diffs < 0, axis=1)

    # Evitar división por cero en ventanas sin variabilidad
    suma_porta = n_above + n_below
    porta_index = np.divide(n_below, suma_porta, out=np.zeros_like(n_below, dtype=float), where=suma_porta!=0)

    # ---- Guzik Index ----
    d_above = np.sum(np.abs(diffs) * (diffs > 0), axis=1) / np.sqrt(2)
    d_total = np.sum(np.abs(diffs), axis=1) / np.sqrt(2)
    guzic_index = np.divide(d_above, d_total, out=np.zeros_like(d_above, dtype=float), where=d_total!=0)

    # Máscaras booleanas de diferencias
    mask_50 = np.abs(diffs) > 50
    mask_20 = np.abs(diffs) > 20

    # 5. Calcular features sobre las diferencias (ventana CORTA)
    nn50 = np.sum(mask_50, axis=1)
    nn20 = np.sum(mask_20, axis=1)
    sdsd = np.std(diffs, axis=1)
    sd1 = np.sqrt((sdsd**2) / 2) # variabilidad a corto plazo (Poincaré SD1)

    # 6. Calcular features sobre los valores absolutos (ventana CORTA)
    mean_val = np.mean(X_ventanas_short, axis=1)
    std_val = np.std(X_ventanas_short, axis=1)
    var_val = std_val ** 2

    # 7. Calcular features sobre la ventana LARGA
    std_long = np.std(X_ventanas_long, axis=1)
    # Evitar raíces negativas limitando el valor mínimo a 0
    inner_value = 2 * std_long**2 - sd1**2
    sd2 = np.sqrt(np.maximum(inner_value, 0))  # variabilidad a largo plazo (Poincaré SD2)
    c_n = np.pi * sd1 * sd2  # área del elipse de Poincaré

    # 8. Calcular CCM sobre la ventana CORTA
    ventanas_4puntos = sliding_window_view(X_ventanas_short, window_shape=4, axis=1)
    rr_i = ventanas_4puntos[:, 0]
    rr_i1 = ventanas_4puntos[:, 1]
    rr_i2 = ventanas_4puntos[:, 2]
    rr_i3 = ventanas_4puntos[:, 3]

    areas = 0.5 * np.abs(rr_i * (rr_i2 - rr_i3) - rr_i1 * (rr_i1 - rr_i3) + rr_i2 * (rr_i1 - rr_i2))

    denominador_ccm = c_n * (window_size - 2)
    ccm = np.divide(np.sum(areas, axis=1), denominador_ccm, out=np.zeros_like(c_n), where=denominador_ccm!=0)
    ccm = np.where(ccm > 1, 1, ccm)

    # 9. Crear el DataFrame combinando todo

    # NUEVO: Generar dinámicamente un diccionario con cada RR de la ventana corta
    rr_columns = {f'rr_{i+1}': X_ventanas_short[:, i] for i in range(window_size)}

    # Diccionario con el resto de tus features estadísticas
    stats_columns = {
        'n_above': n_above,
        'n_below': n_below,
        'nn20': nn20,
        'nn50': nn50,
        'sdsd': sdsd,
        'mean': mean_val,
        'std': std_val,
        'var': var_val,
        'std_long': std_long,
        'sd1': sd1,
        'sd2': sd2,
        'c_n': c_n,
        'ccm': ccm,
        'porta': porta_index,
        'guzik': guzic_index,
        'target': y_target
    }

    # Combinamos ambos diccionarios para crear el DataFrame final
    # El operador ** desempaqueta los diccionarios
    df = pd.DataFrame({**rr_columns, **stats_columns})

    return df

In [4]:
def extract_hrv_features_2(serie, window_size=20, window_size_long=40):
    """
    Toma una serie de intervalos RR y los tamaños de ventana para construir
    un DataFrame de características ortogonales y la variable objetivo (target).
    """
    if window_size_long < window_size:
        raise ValueError("window_size_long debe ser mayor o igual a window_size")

    serie = np.asarray(serie, dtype=float)

    # 1. Generar ventanas sobre la serie COMPLETA
    ventanas_long = sliding_window_view(serie, window_size_long)

    # 2. Separar variables predictoras del target
    X_ventanas_long = ventanas_long[:-1]
    y_target = serie[window_size_long:]

    # 3. Extraer ventana CORTA
    X_ventanas_short = X_ventanas_long[:, -window_size:]

    # 4. Calcular diferencias dentro de la ventana CORTA
    diffs = np.diff(X_ventanas_short, axis=1)

    # ---- MÉTRICAS TRADICIONALES ----
    n_above = np.sum(diffs > 0, axis=1)
    n_below = np.sum(diffs < 0, axis=1)
    suma_porta = n_above + n_below
    porta_index = np.divide(n_below, suma_porta, out=np.zeros_like(n_below, dtype=float), where=suma_porta!=0)

    d_above = np.sum(np.abs(diffs) * (diffs > 0), axis=1) / np.sqrt(2)
    d_total = np.sum(np.abs(diffs), axis=1) / np.sqrt(2)
    guzic_index = np.divide(d_above, d_total, out=np.zeros_like(d_above, dtype=float), where=d_total!=0)

    nn50 = np.sum(np.abs(diffs) > 50, axis=1)
    nn20 = np.sum(np.abs(diffs) > 20, axis=1)

    sdsd = np.std(diffs, axis=1)
    sd1 = np.sqrt((sdsd**2) / 2)

    mean_val = np.mean(X_ventanas_short, axis=1)
    std_val = np.std(X_ventanas_short, axis=1)
    var_val = std_val ** 2

    std_long = np.std(X_ventanas_long, axis=1)
    inner_value = 2 * std_long**2 - sd1**2
    sd2 = np.sqrt(np.maximum(inner_value, 0))
    c_n = np.pi * sd1 * sd2

    # CCM
    ventanas_4puntos = sliding_window_view(X_ventanas_short, window_shape=4, axis=1)
    rr_i, rr_i1, rr_i2, rr_i3 = ventanas_4puntos[:, 0], ventanas_4puntos[:, 1], ventanas_4puntos[:, 2], ventanas_4puntos[:, 3]
    areas = 0.5 * np.abs(rr_i * (rr_i2 - rr_i3) - rr_i1 * (rr_i1 - rr_i3) + rr_i2 * (rr_i1 - rr_i2))
    denominador_ccm = c_n * (window_size - 2)
    ccm = np.divide(np.sum(areas, axis=1), denominador_ccm, out=np.zeros_like(c_n), where=denominador_ccm!=0)
    ccm = np.where(ccm > 1, 1, ccm)

    # -------------------------------------------------------------
    # NUEVAS CARACTERÍSTICAS ORTOGONALES (NO COLINEALES)
    # -------------------------------------------------------------

    # A. Coeficiente de Variación (CV)
    cv = np.divide(std_val, mean_val, out=np.zeros_like(std_val, dtype=float), where=mean_val!=0)

    # B. Robustez a Outliers: Rango Intercuartílico (IQR) y MAD
    q75, q25 = np.percentile(X_ventanas_short, [75, 25], axis=1)
    iqr = q75 - q25

    median_val = np.median(X_ventanas_short, axis=1)
    mad = np.median(np.abs(X_ventanas_short - median_val[:, None]), axis=1)

    # C. Fragmentación del Ritmo Cardíaco (PIP - Puntos de Inflexión)
    diffs_1 = diffs[:, :-1]
    diffs_2 = diffs[:, 1:]
    inflections = (diffs_1 * diffs_2) <= 0
    pip = np.sum(inflections, axis=1) / (window_size - 2)

    # D. Asimetría (Skewness) de las diferencias
    mean_diffs = np.mean(diffs, axis=1, keepdims=True)
    std_diffs = np.std(diffs, axis=1, keepdims=True)
    std_diffs_safe = np.where(std_diffs == 0, 1e-10, std_diffs) # Evitar NaN
    skewness = np.mean(((diffs - mean_diffs) / std_diffs_safe)**3, axis=1)

    # -------------------------------------------------------------
    # EMPAQUETADO FINAL
    # -------------------------------------------------------------
    rr_columns = {f'rr_{i+1}': X_ventanas_short[:, i] for i in range(window_size)}

    stats_columns = {
        'n_above': n_above, 'n_below': n_below, 'nn20': nn20, 'nn50': nn50,
        'sdsd': sdsd, 'mean': mean_val, 'std': std_val, 'var': var_val,
        'std_long': std_long, 'sd1': sd1, 'sd2': sd2, 'c_n': c_n,
        'ccm': ccm, 'porta': porta_index, 'guzik': guzic_index,
        'cv': cv, 'iqr': iqr, 'mad': mad, 'pip': pip, 'skewness': skewness, # <--- NUEVAS
        'target': y_target
    }

    df = pd.DataFrame({**rr_columns, **stats_columns})
    return df

In [5]:
# Cargar los datos
series_list = '/content/drive/MyDrive/Challenges_ML-DL/HRV_prediction/series/'
files = ['006.txt',
        '16265.txt',
        '16273.txt',
        '16786.txt',
        '16795.txt',
        '17453.txt',
        '18177.txt',
        '18184.txt',
        '16539.txt',
        'nsr054RRcl.txt',
        '005.txt',
        '16420.txt',
        '19140.txt',
        'nsr047RRcl.txt',
        'nsr048RRcl.txt',
        'nsr049RRcl.txt',
        '008.txt',
        '009.txt',
        '013.txt']
window_size = 20
np.random.seed(42)
# select 50 random files
files_train = np.random.choice(files, size=11, replace=False)

# the rest of the files will be used for testing
files_test = [f for f in files if f not in files_train]

# Crear DataFrames para entrenamiento y prueba
df_train = pd.DataFrame()
for file in files_train:
    data = np.loadtxt(os.path.join(series_list, file), dtype=int)
    df_temp = extract_hrv_features_2(data, window_size, window_size_long=40)

    # NUEVO: Agregamos una columna con el nombre del archivo para usarla en la estratificación
    df_temp['file_id'] = file

    df_train = pd.concat([df_train, df_temp], ignore_index=True)

# 1. Separar features (X), target (y) y grupos (file_id)
# Es VITAL quitar 'file_id' de X para que el modelo no lo use como feature predictivo
X = df_train.drop(columns=['target', 'file_id'])
y = df_train['target']
groups = df_train['file_id'] # Nuestra variable de estratificación para el K-Fold training

In [6]:
# 1. Definir qué vamos a usar para evitar la colinealidad
# Selecciona las features más representativas (ortogonales)
feature_cols = ['mean', 'sdsd', 'sd2', 'ccm', 'guzik', 'nn50',  'porta', 'std', 'c_n', 'cv', 'iqr', 'mad', 'pip', 'skewness']

# Seleccionamos los últimos 3 latidos de la ventana de tamaño 20
lag_cols = ['rr_20', 'rr_19', 'rr_18']

X_feats = df_train[feature_cols].values
X_lags = df_train[lag_cols].values
y = df_train['target'].values
groups = df_train['file_id']

scaler_feats = StandardScaler()
X_feats_scaled = scaler_feats.fit_transform(X_feats).astype('float32')

scaler_lags = StandardScaler()
X_lags_scaled = scaler_lags.fit_transform(X_lags).astype('float32')

y_scaler = StandardScaler()
y_scaled = y_scaler.fit_transform(y.reshape(-1, 1)).flatten().astype('float32')

# 3. Factorizar grupos para la jerarquía
group_idx, group_names = pd.factorize(groups)

In [7]:
# 1. Empaquetar TODOS los datos juntos (CRÍTICO para mantener la alineación de las filas)
# Asegúrate de que todas sean matrices 2D (shape N, cols)
dataset_all = np.column_stack([
    X_feats_scaled,       # Índices de columnas: 0 a 5 (si son 6 features)
    X_lags_scaled,        # Índices de columnas: 6 a 8 (si son 3 lags)
    y_scaled,             # Índice: 9
    group_idx             # Índice: 10
])

N_total_rows = dataset_all.shape[0]
n_feats = X_feats_scaled.shape[1]
n_lags = X_lags_scaled.shape[1]

batch_size = 2048

with pm.Model() as hrv_minibatch_model:

    hrv_minibatch_model.add_coord("sujetos", group_names)
    hrv_minibatch_model.add_coord("features", feature_cols)
    hrv_minibatch_model.add_coord("lags", lag_cols)

    # 1. El Minibatch unificado
    minibatch = pm.Minibatch(dataset_all, batch_size=batch_size)

    # Desempaquetado dinámico (PyMC ya no se quejará de estos cortes)
    X_feats_mb = minibatch[:, 0:n_feats]
    X_lags_mb  = minibatch[:, n_feats:(n_feats + n_lags)]
    y_mb       = minibatch[:, (n_feats + n_lags)]
    idx_mb     = pt.cast(minibatch[:, (n_feats + n_lags + 1)], 'int32')

    # 2. Intercepto Jerárquico
    mu_alpha = pm.Normal("mu_alpha", mu=0, sigma=1)
    sigma_alpha = pm.HalfNormal("sigma_alpha", sigma=1)
    z_alpha = pm.Normal("z_alpha", mu=0, sigma=1, dims="sujetos")
    alpha = pm.Deterministic("alpha", mu_alpha + z_alpha * sigma_alpha, dims="sujetos")

    # 3. Componentes Estadísticos y AR
    beta_feats = pm.Normal("beta_feats", mu=0, sigma=1, dims="features")
    rho = pm.Normal("rho", mu=0, sigma=0.5, dims="lags")

    # 4. Ecuación Combinada
    mu_y = alpha[idx_mb] + pm.math.sum(X_feats_mb * beta_feats, axis=1) + pm.math.sum(X_lags_mb * rho, axis=1)

    sigma_error = pm.HalfNormal("sigma_error", sigma=1)
    nu = pm.Gamma("nu", alpha=2, beta=0.1) + 2

    # --- LA MAGIA: REEMPLAZO DE y_obs ---

    # A. Creamos la distribución teórica de nuestro target
    dist_y = pm.StudentT.dist(nu=nu, mu=mu_y, sigma=sigma_error)

    # B. Calculamos la probabilidad logarítmica de las observaciones (y_mb) contra esa distribución
    logp_mb = pm.logp(dist_y, y_mb)

    # C. Inyectamos esto al modelo usando Potential.
    # Es VITAL multiplicar por (N_total_rows / batch_size) para que el optimizador
    # sepa que este pequeño lote representa a todo el inmenso dataset.
    pm.Potential("y_obs_potential", pt.sum(logp_mb) * (N_total_rows / batch_size))

# --- Ajuste ---
with hrv_minibatch_model:
    print(f"Iniciando ADVI con minibatches y Potential. Filas totales: {N_total_rows}...")

    mean_field = pm.fit(
        n=50000,
        method='advi',
        obj_optimizer=pm.adam(learning_rate=0.01),
        progressbar=True
    )

    print("Muestreando de la aproximación...")
    trace = mean_field.sample(2000)

Iniciando ADVI con minibatches y Potential. Filas totales: 1044950...


Output()

Muestreando de la aproximación...


In [8]:
# Crear DataFrame para prueba
df_test = pd.DataFrame()
for file in files_test:
    data = np.loadtxt(os.path.join(series_list, file), dtype=int)
    df_temp = extract_hrv_features_2(data, window_size, window_size_long=40)
    df_temp['file_id'] = file
    df_test = pd.concat([df_test, df_temp], ignore_index=True)

# Separar variables correspondientes
X_feats_test = df_test[feature_cols].values
X_lags_test = df_test[lag_cols].values
y_test_raw = df_test['target'].values
groups_test = df_test['file_id']

# ESCALAR usando los mismos objetos StandardScaler del entrenamiento
# (Asumiendo que guardaste scaler_feats, scaler_lags y y_scaler)
X_feats_test_scaled = scaler_feats.transform(X_feats_test).astype('float32')
X_lags_test_scaled = scaler_lags.transform(X_lags_test).astype('float32')

In [9]:
import gc

# 1. Extraer los parámetros del trace (esto se mantiene igual)
alpha_samples = trace.posterior["alpha"].stack(sample=("chain", "draw")).values.astype(np.float32)
mu_alpha_samples = trace.posterior["mu_alpha"].stack(sample=("chain", "draw")).values.astype(np.float32)
beta_feats_samples = trace.posterior["beta_feats"].stack(sample=("chain", "draw")).values.astype(np.float32)
rho_samples = trace.posterior["rho"].stack(sample=("chain", "draw")).values.astype(np.float32)

n_samples = alpha_samples.shape[1]
N_test = X_feats_test_scaled.shape[0]

# 2. Mapeo Vectorizado de Sujetos (¡Mucho más rápido que el for loop original!)
# Creamos un diccionario para buscar rápido el índice
group_to_idx = {name: idx for idx, name in enumerate(group_names)}

# Array con el índice correspondiente. Usamos -1 para sujetos "no vistos"
alpha_indices = np.array([group_to_idx.get(g, -1) for g in groups_test])

# 3. Preparar arrays ligeros para guardar SÓLO los resultados finales (poco peso en RAM)
y_pred_mean_ms = np.zeros(N_test, dtype=np.float32)
# y_pred_std_ms = np.zeros(N_test, dtype=np.float32) # Descomenta si quieres la incertidumbre

# 4. Procesamiento por Lotes (Chunking) para salvar la RAM
batch_size = 50000 # Si vuelve a fallar, reduce este número a 25000

print(f"Iniciando predicciones en lotes. Total de filas: {N_test}")

for start_idx in range(0, N_test, batch_size):
    end_idx = min(start_idx + batch_size, N_test)
    batch_len = end_idx - start_idx

    # Extraer el bloque actual
    X_feats_batch = X_feats_test_scaled[start_idx:end_idx]
    X_lags_batch = X_lags_test_scaled[start_idx:end_idx]
    idx_batch = alpha_indices[start_idx:end_idx]

    # Crear la matriz alpha SÓLO para este pequeño lote
    alpha_batch_matrix = np.zeros((batch_len, n_samples), dtype=np.float32)

    # Rellenar los alphas del lote
    for i, g_idx in enumerate(idx_batch):
        if g_idx != -1:
            alpha_batch_matrix[i, :] = alpha_samples[g_idx, :]
        else:
            alpha_batch_matrix[i, :] = mu_alpha_samples

    # Ecuación predictiva lineal para este lote
    pred_scaled_batch = (
        alpha_batch_matrix +
        np.dot(X_feats_batch, beta_feats_samples) +
        np.dot(X_lags_batch, rho_samples)
    )

    # Desescalar este lote a milisegundos
    pred_ms_batch = y_scaler.inverse_transform(pred_scaled_batch)

    # Guardar el resultado agregado (colapsamos las 2000 columnas a 1 sola)
    y_pred_mean_ms[start_idx:end_idx] = np.mean(pred_ms_batch, axis=1)

    # Liberar memoria explícitamente
    del alpha_batch_matrix, pred_scaled_batch, pred_ms_batch
    gc.collect()

    print(f"Lote procesado: {end_idx} / {N_test}")

print("¡Predicciones terminadas exitosamente!")

Iniciando predicciones en lotes. Total de filas: 804505
Lote procesado: 50000 / 804505
Lote procesado: 100000 / 804505
Lote procesado: 150000 / 804505
Lote procesado: 200000 / 804505
Lote procesado: 250000 / 804505
Lote procesado: 300000 / 804505
Lote procesado: 350000 / 804505
Lote procesado: 400000 / 804505
Lote procesado: 450000 / 804505
Lote procesado: 500000 / 804505
Lote procesado: 550000 / 804505
Lote procesado: 600000 / 804505
Lote procesado: 650000 / 804505
Lote procesado: 700000 / 804505
Lote procesado: 750000 / 804505
Lote procesado: 800000 / 804505
Lote procesado: 804505 / 804505
¡Predicciones terminadas exitosamente!


In [11]:
# --- ¡ATENCIÓN! ---
# No necesitas desescalar ni calcular np.mean() aquí.
# La variable 'y_pred_mean_ms' ya salió lista y en milisegundos del bucle anterior.

# --- EVALUACIÓN DE MÉTRICAS ---
mae = mean_absolute_error(y_test_raw, y_pred_mean_ms)
rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_mean_ms))
r2 = r2_score(y_test_raw, y_pred_mean_ms)

print(f"Métricas en Test de No Vistos:")
print(f"MAE: {mae:.2f} ms")
print(f"RMSE: {rmse:.2f} ms")
print(f"R²: {r2:.4f}")

# --- EVALUACIÓN DEL OBJETIVO DE NEGOCIO ---
# Tolerancia de +- 15 ms
errores_absolutos = np.abs(y_test_raw - y_pred_mean_ms)
latidos_en_tolerancia = np.mean(errores_absolutos <= 15.0) * 100

print(f"\nPorcentaje de intervalos RR predichos dentro de la tolerancia (±15ms): {latidos_en_tolerancia:.2f}%")

Métricas en Test de No Vistos:
MAE: 23.31 ms
RMSE: 35.18 ms
R²: 0.9398

Porcentaje de intervalos RR predichos dentro de la tolerancia (±15ms): 46.57%
